## Gerar Exceis

In [1]:
# ============================================================
# DEGRADAÇÃO TEMPORAL REALISTA DE CORROSÃO
# - Só cresce anomalias com coluna C == "corrosion"
# - "damage" e "water" mantêm-se inalteradas
# - Gera 20 ficheiros Excel (de 6 em 6 meses, durante 10 anos)
# - Crescimento desacelerado ao longo do tempo
# ============================================================

import re
import math
from pathlib import Path

import numpy as np
import cv2
from openpyxl import load_workbook
from PIL import Image

# ============================================================
# CONFIG
# ============================================================

# Pasta onde está o excel original
PASTA_BASE = r"base_excel"
FICHEIRO_EXCEL = "metricas_anomalias_faces_malha.xlsx"

# Pasta das imagens base para obter largura/altura corretas
PASTA_IMAGENS = "image"

# Pasta onde serão guardados os 20 Excels
PASTA_SAIDA_EXCELS = 'excel_result/degradacao_temporal_excels'

# ------------------------------------------------------------
# PARÂMETROS TEMPORAIS
# ------------------------------------------------------------
NUM_PASSOS = 20              # 20 excels
MESES_POR_PASSO = 6          # de 6 em 6 meses

# ------------------------------------------------------------
# PARÂMETROS DE DEGRADAÇÃO
# ------------------------------------------------------------
# Kernel da dilatação
KERNEL_SIZE = 3

# "Força" inicial de degradação por passo.
# Não precisa de ser inteira. O acumulador trata das frações.
BASE_STEP_STRENGTH = 1.8

# Quanto maior, mais rapidamente a degradação abranda.
# Valores típicos: 1.5 a 4.0
DECAY_RATE = 2.8

# Mínimo de crescimento residual relativo no último passo
# evita que a degradação "morra completamente"
# Ex.: 0.15 = no fim ainda há 15% da força inicial
MIN_RELATIVE_GROWTH = 0.15

# Se True, aplica fechamento morfológico leve após dilatação
# para tapar pequenas falhas/aberturas e deixar o dano mais coeso
APPLY_MORPH_CLOSE = True
MORPH_CLOSE_KERNEL_SIZE = 3

AREA_DECIMALS = 6
EXTENSOES_VALIDAS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp", ".webp"}

# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def parse_polygon_string(poly_str):
    if poly_str is None:
        return []

    poly_str = str(poly_str).strip()
    if not poly_str:
        return []

    matches = re.findall(r"\[\s*(-?\d+(?:\.\d+)?)\s*,\s*(-?\d+(?:\.\d+)?)\s*\]", poly_str)
    return [(float(x), float(y)) for x, y in matches]


def polygon_to_string(points):
    return "//".join(f"[{int(round(x))},{int(round(y))}]" for x, y in points)


def polygon_area_pixels(points):
    if len(points) < 3:
        return 0.0

    pts = np.array(points, dtype=float)
    x = pts[:, 0]
    y = pts[:, 1]
    area = 0.5 * abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))
    return float(area)


def safe_float(v, default=0.0):
    try:
        return float(v)
    except:
        return default


def normalize_anomaly_type(v):
    if v is None:
        return ""
    return str(v).strip().lower()


def index_images_by_name(root_folder):
    root = Path(root_folder)
    image_map = {}
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in EXTENSOES_VALIDAS:
            image_map[p.name.lower()] = p
    return image_map


def get_image_size_cached(image_name, image_index, size_cache):
    if image_name in size_cache:
        return size_cache[image_name]

    img_path = image_index.get(image_name)
    if img_path is None:
        return None

    with Image.open(img_path) as img_ref:
        size_cache[image_name] = img_ref.size  # (width, height)

    return size_cache[image_name]


def polygon_to_mask(points, width, height):
    mask = np.zeros((height, width), dtype=np.uint8)

    pts = np.array(
        [[
            [
                min(max(int(round(x)), 0), width - 1),
                min(max(int(round(y)), 0), height - 1)
            ]
            for x, y in points
        ]],
        dtype=np.int32
    )

    if pts.shape[1] >= 3:
        cv2.fillPoly(mask, pts, 255)

    return mask


def dilate_mask(mask, kernel_size=3, iterations=1):
    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    dilated = cv2.dilate(mask, kernel, iterations=iterations)
    return dilated


def close_mask(mask, kernel_size=3):
    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    return cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)


def mask_to_polygon(mask):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return []

    contour = max(contours, key=cv2.contourArea)

    epsilon = 0.002 * cv2.arcLength(contour, True)
    approx = cv2.approxPolyDP(contour, epsilon, True)

    points = [(float(pt[0][0]), float(pt[0][1])) for pt in approx]
    return points


def get_step_strength(step_idx, total_steps, base_strength, decay_rate, min_relative_growth):
    """
    step_idx: 1..total_steps
    Retorna força relativa decrescente ao longo do tempo.
    No início ~1.0
    No fim ~min_relative_growth
    """
    if total_steps <= 1:
        return base_strength

    s = (step_idx - 1) / (total_steps - 1)  # 0..1

    # exponencial decrescente normalizada
    relative = math.exp(-decay_rate * s)

    # remap para nunca cair abaixo de min_relative_growth
    relative = min_relative_growth + (1.0 - min_relative_growth) * relative

    return base_strength * relative


def build_output_filename(base_name, step_idx, months_elapsed):
    stem = Path(base_name).stem
    suffix = Path(base_name).suffix
    years = months_elapsed // 12
    rem_months = months_elapsed % 12
    return f"{stem}_degradacao_t{step_idx:02d}_{years:02d}a_{rem_months:02d}m{suffix}"


# ============================================================
# PREPARAÇÃO
# ============================================================

excel_path = Path(PASTA_BASE) / FICHEIRO_EXCEL
output_dir = Path(PASTA_SAIDA_EXCELS)
output_dir.mkdir(parents=True, exist_ok=True)

if not excel_path.exists():
    raise FileNotFoundError(f"Excel não encontrado: {excel_path}")

image_index = index_images_by_name(PASTA_IMAGENS)
if not image_index:
    raise RuntimeError(f"Não foram encontradas imagens em: {PASTA_IMAGENS}")

print(f"[INFO] Excel base: {excel_path}")
print(f"[INFO] Pasta imagens: {PASTA_IMAGENS}")
print(f"[INFO] Pasta saída excels: {output_dir}")
print(f"[INFO] Imagens indexadas: {len(image_index)}")

# cache de tamanhos de imagem
image_size_cache = {}

# workbook que vai sendo degradado cumulativamente
wb = load_workbook(excel_path)

# acumulador por anomalia para permitir forças fracionárias
# chave única por linha: (sheet_name, row)
growth_budget = {}

# ============================================================
# CICLO TEMPORAL
# ============================================================

for step in range(1, NUM_PASSOS + 1):
    months_elapsed = step * MESES_POR_PASSO

    # força deste semestre
    step_strength = get_step_strength(
        step_idx=step,
        total_steps=NUM_PASSOS,
        base_strength=BASE_STEP_STRENGTH,
        decay_rate=DECAY_RATE,
        min_relative_growth=MIN_RELATIVE_GROWTH
    )

    print("\n" + "=" * 68)
    print(f"[PASSO {step:02d}/{NUM_PASSOS}]  Tempo = {months_elapsed} meses")
    print(f"[INFO] Força teórica deste passo = {step_strength:.4f}")
    print("=" * 68)

    total_corrosion_updated = 0
    total_non_corrosion_kept = 0
    total_skipped = 0

    for ws in wb.worksheets:
        print(f"\n--- Folha: {ws.title} ---")

        corrosion_updated_sheet = 0
        non_corrosion_kept_sheet = 0
        skipped_sheet = 0

        for row in range(2, ws.max_row + 1):
            anomaly_type = normalize_anomaly_type(ws[f"C{row}"].value)
            polygon_str = ws[f"E{row}"].value
            area_cm2_old = ws[f"F{row}"].value
            image_path_excel = ws[f"G{row}"].value

            # Se não houver geometria ou imagem associada, ignora
            if not polygon_str or not image_path_excel:
                skipped_sheet += 1
                total_skipped += 1
                continue

            points = parse_polygon_string(polygon_str)
            if len(points) < 3:
                skipped_sheet += 1
                total_skipped += 1
                continue

            # Só corrosão cresce
            if anomaly_type != "corrosion":
                non_corrosion_kept_sheet += 1
                total_non_corrosion_kept += 1
                continue

            image_name = Path(str(image_path_excel)).name.lower()
            size = get_image_size_cached(image_name, image_index, image_size_cache)

            if size is None:
                skipped_sheet += 1
                total_skipped += 1
                continue

            width, height = size

            old_pixel_area = polygon_area_pixels(points)
            old_cm2 = safe_float(area_cm2_old, default=0.0)

            if old_pixel_area <= 0:
                skipped_sheet += 1
                total_skipped += 1
                continue

            row_key = (ws.title, row)
            prev_budget = growth_budget.get(row_key, 0.0)
            new_budget = prev_budget + step_strength

            # quantas iterações inteiras aplicar neste passo
            iterations_to_apply = int(math.floor(new_budget))
            growth_budget[row_key] = new_budget - iterations_to_apply

            # Se neste semestre ainda não atingiu 1 iteração,
            # mantém como está e passa orçamento sobrante ao próximo passo
            if iterations_to_apply <= 0:
                corrosion_updated_sheet += 1
                total_corrosion_updated += 1
                continue

            # 1) polígono -> máscara
            mask = polygon_to_mask(points, width, height)

            # 2) dilatar máscara
            grown_mask = dilate_mask(
                mask,
                kernel_size=KERNEL_SIZE,
                iterations=iterations_to_apply
            )

            # 3) opcional: fechar pequenas falhas
            if APPLY_MORPH_CLOSE:
                grown_mask = close_mask(grown_mask, kernel_size=MORPH_CLOSE_KERNEL_SIZE)

            # 4) máscara -> novo polígono
            new_points = mask_to_polygon(grown_mask)

            if len(new_points) < 3:
                skipped_sheet += 1
                total_skipped += 1
                continue

            new_pixel_area = polygon_area_pixels(new_points)

            # mesma conversão implícita pixel² -> cm² do registo original
            if old_cm2 > 0:
                cm2_per_pixel2 = old_cm2 / old_pixel_area
                new_cm2 = new_pixel_area * cm2_per_pixel2
            else:
                new_cm2 = 0.0

            # atualizar Excel em memória
            ws[f"E{row}"] = polygon_to_string(new_points)
            ws[f"F{row}"] = round(new_cm2, AREA_DECIMALS)

            corrosion_updated_sheet += 1
            total_corrosion_updated += 1

        print(f"Corrosion processadas:      {corrosion_updated_sheet}")
        print(f"Damage/Water mantidas:      {non_corrosion_kept_sheet}")
        print(f"Ignoradas por erro/falta:   {skipped_sheet}")

    # guardar snapshot deste instante temporal
    output_name = build_output_filename(
        base_name=FICHEIRO_EXCEL,
        step_idx=step,
        months_elapsed=months_elapsed
    )
    output_path = output_dir / output_name
    wb.save(output_path)

    print("\n[SNAPSHOT GUARDADO]")
    print(f"Ficheiro: {output_path}")
    print(f"Corrosion processadas: {total_corrosion_updated}")
    print(f"Damage/Water mantidas: {total_non_corrosion_kept}")
    print(f"Ignoradas:             {total_skipped}")

print("\n" + "=" * 68)
print("PROCESSO COMPLETO")
print("=" * 68)
print(f"Excel base: {excel_path}")
print(f"Excels gerados: {NUM_PASSOS}")
print(f"Pasta de saída: {output_dir}")

[INFO] Excel base: base_excel\metricas_anomalias_faces_malha.xlsx
[INFO] Pasta imagens: image
[INFO] Pasta saída excels: excel_result\degradacao_temporal_excels
[INFO] Imagens indexadas: 831

[PASSO 01/20]  Tempo = 6 meses
[INFO] Força teórica deste passo = 1.8000

--- Folha: Anomalies00 ---
Corrosion processadas:      2191
Damage/Water mantidas:      662
Ignoradas por erro/falta:   0

--- Folha: corrosion ---
Corrosion processadas:      2191
Damage/Water mantidas:      0
Ignoradas por erro/falta:   1

--- Folha: damage ---
Corrosion processadas:      0
Damage/Water mantidas:      651
Ignoradas por erro/falta:   1

--- Folha: water ---
Corrosion processadas:      0
Damage/Water mantidas:      11
Ignoradas por erro/falta:   1

[SNAPSHOT GUARDADO]
Ficheiro: excel_result\degradacao_temporal_excels\metricas_anomalias_faces_malha_degradacao_t01_00a_06m.xlsx
Corrosion processadas: 4382
Damage/Water mantidas: 1324
Ignoradas:             3

[PASSO 02/20]  Tempo = 12 meses
[INFO] Força teórica 

## Gerar Exceis - c/ tendência

In [7]:
# ============================================================
# DEGRADAÇÃO TEMPORAL DE CORROSÃO ORIENTADA PELA FORMA ORIGINAL
# ============================================================

import re
import math
from dataclasses import dataclass
from pathlib import Path

import cv2
import numpy as np
from openpyxl import load_workbook
from PIL import Image


# ============================================================
# CONFIG
# ============================================================

PASTA_BASE = r"base_excel"
FICHEIRO_EXCEL = "metricas_anomalias_faces_malha.xlsx"
PASTA_IMAGENS = "image"
PASTA_SAIDA_EXCELS = 'excel_result/degradacao_temporal_excels/trend'

NUM_PASSOS = 20
MESES_POR_PASSO = 6

BASE_STEP_STRENGTH = 1.8
DECAY_RATE = 2.8
MIN_RELATIVE_GROWTH = 0.15

BASE_KERNEL_SIZE = 3
MAX_EXTRA_MAJOR_RADIUS = 3
ELONGATION_RATIO_FOR_MAX_GROWTH = 4.0
TREND_RESPONSE_EXPONENT = 0.70
MIN_PIXELS_FOR_TREND_ANALYSIS = 25
MAX_ELONGATION_RATIO = 20.0

APPLY_MORPH_CLOSE = True
MORPH_CLOSE_KERNEL_SIZE = 3
POLYGON_SIMPLIFICATION_FACTOR = 0.0015

AREA_DECIMALS = 6

EXTENSOES_VALIDAS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp", ".webp"}


@dataclass
class CorrosionTrend:
    angle_deg: float
    elongation_ratio: float
    trend_strength: float
    major_radius: int
    minor_radius: int
    kernel: np.ndarray
    original_pixel_area: float
    original_area_cm2: float
    cm2_per_pixel2: float


def parse_polygon_string(poly_str):
    if not poly_str:
        return []
    matches = re.findall(r"\[\s*(-?\d+(?:\.\d+)?)\s*,\s*(-?\d+(?:\.\d+)?)\s*\]", str(poly_str))
    return [(float(x), float(y)) for x, y in matches]


def polygon_to_string(points):
    return "//".join(f"[{int(round(x))},{int(round(y))}]" for x, y in points)


def polygon_area_pixels(points):
    if len(points) < 3:
        return 0.0
    pts = np.asarray(points, dtype=np.float64)
    return float(0.5 * abs(np.dot(pts[:, 0], np.roll(pts[:, 1], -1)) - np.dot(pts[:, 1], np.roll(pts[:, 0], -1))))


def safe_float(value, default=0.0):
    try:
        return float(value)
    except (TypeError, ValueError):
        return default


def normalize_anomaly_type(value):
    if value is None:
        return ""
    return str(value).strip().lower()


def index_images_by_name(root_folder):
    root = Path(root_folder)
    image_map = {}
    for path in root.rglob("*"):
        if path.is_file() and path.suffix.lower() in EXTENSOES_VALIDAS:
            image_map[path.name.lower()] = path
    return image_map


def find_matching_image_size(image_path_excel, image_index, size_cache):
    if not image_path_excel:
        return None

    raw_str = str(image_path_excel).strip()
    name_lower = Path(raw_str).name.lower()

    # 1. Tenta correspondência exata pelo nome do ficheiro
    if name_lower in image_index:
        img_path = image_index[name_lower]
        if name_lower not in size_cache:
            try:
                with Image.open(img_path) as img:
                    size_cache[name_lower] = img.size
            except Exception:
                return None
        return size_cache[name_lower]

    # 2. Tenta encontrar por subcadência caso o nome no Excel tenha sufixos/prefixos diferentes
    for indexed_name, img_path in image_index.items():
        if indexed_name in name_lower or name_lower in indexed_name:
            if indexed_name not in size_cache:
                try:
                    with Image.open(img_path) as img:
                        size_cache[indexed_name] = img.size
                except Exception:
                    continue
            return size_cache[indexed_name]

    return None


def polygon_to_mask(points, width, height):
    mask = np.zeros((height, width), dtype=np.uint8)
    if len(points) < 3:
        return mask
    clipped = [[min(max(int(round(x)), 0), width - 1), min(max(int(round(y)), 0), height - 1)] for x, y in points]
    cv2.fillPoly(mask, [np.asarray(clipped, dtype=np.int32).reshape((-1, 1, 2))], 255)
    return mask


def mask_to_polygon(mask):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not contours:
        return []
    contour = max(contours, key=cv2.contourArea)
    approx = cv2.approxPolyDP(contour, POLYGON_SIMPLIFICATION_FACTOR * cv2.arcLength(contour, True), True)
    return [(float(pt[0][0]), float(pt[0][1])) for pt in approx]


def close_mask(mask, kernel_size=3):
    if kernel_size < 1:
        return mask
    if kernel_size % 2 == 0:
        kernel_size += 1
    return cv2.morphologyEx(mask, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size)))


def analyse_shape_trend(mask):
    pixel_count = int(cv2.countNonZero(mask))
    if pixel_count < MIN_PIXELS_FOR_TREND_ANALYSIS:
        return 0.0, 1.0, 0.0
    moments = cv2.moments(mask, binaryImage=True)
    m00 = moments["m00"]
    if m00 <= 0:
        return 0.0, 1.0, 0.0
    cov = np.array([[moments["mu20"] / m00, moments["mu11"] / m00], [moments["mu11"] / m00, moments["mu02"] / m00]], dtype=np.float64)
    evals, evecs = np.linalg.eigh(cov)
    order = np.argsort(evals)[::-1]
    major_v = evecs[:, order[0]]
    angle_deg = math.degrees(math.atan2(major_v[1], major_v[0])) % 180.0
    elongation = max(1.0, min(math.sqrt(max(float(evals[order[0]]), 1e-9) / max(float(evals[order[1]]), 1e-9)), MAX_ELONGATION_RATIO))
    trend_str = float(np.clip((elongation - 1.0) / (ELONGATION_RATIO_FOR_MAX_GROWTH - 1.0), 0.0, 1.0)) ** TREND_RESPONSE_EXPONENT
    return angle_deg, elongation, trend_str


def create_corrosion_trend(original_points, width, height, original_area_cm2):
    orig_mask = polygon_to_mask(original_points, width, height)
    angle_deg, elongation_ratio, trend_strength = analyse_shape_trend(orig_mask)
    base_r = (BASE_KERNEL_SIZE - 1) // 2
    major_r = base_r + int(round(MAX_EXTRA_MAJOR_RADIUS * trend_strength))
    minor_r = base_r

    max_r = max(major_r, minor_r)
    kernel = np.zeros(((2 * max_r) + 1, (2 * max_r) + 1), dtype=np.uint8)
    cv2.ellipse(kernel, center=(max_r, max_r), axes=(max_r if major_r == max_r else major_r, minor_r if minor_r != max_r else minor_r), angle=float(angle_deg), startAngle=0, endAngle=360, color=1, thickness=-1)
    kernel[max_r, max_r] = 1

    pix_area = polygon_area_pixels(original_points)
    cm2_per_pixel2 = (original_area_cm2 / pix_area) if (pix_area > 0 and original_area_cm2 > 0) else 0.0

    return CorrosionTrend(angle_deg, elongation_ratio, trend_strength, major_r, minor_r, kernel, pix_area, original_area_cm2, cm2_per_pixel2)


def get_step_strength(step_idx, total_steps, base_strength, decay_rate, min_relative_growth):
    if total_steps <= 1:
        return base_strength
    return base_strength * (min_relative_growth + (1.0 - min_relative_growth) * math.exp(-decay_rate * ((step_idx - 1) / (total_steps - 1))))


def build_output_filename(base_name, step_idx, months_elapsed):
    base_path = Path(base_name)
    return f"{base_path.stem}_degradacao_t{step_idx:02d}_{months_elapsed // 12:02d}a_{months_elapsed % 12:02d}m{base_path.suffix}"


def main():
    excel_path = Path(PASTA_BASE) / FICHEIRO_EXCEL
    output_dir = Path(PASTA_SAIDA_EXCELS)
    output_dir.mkdir(parents=True, exist_ok=True)

    if not excel_path.exists():
        raise FileNotFoundError(f"Excel não encontrado: {excel_path}")

    image_index = index_images_by_name(PASTA_IMAGENS)
    if not image_index:
        raise RuntimeError(f"Nenhuma imagem encontrada na pasta: {PASTA_IMAGENS}")

    print("=" * 76)
    print("DEGRADAÇÃO TEMPORAL ORIENTADA DA CORROSÃO")
    print("=" * 76)

    size_cache = {}
    wb = load_workbook(excel_path)
    growth_budget = {}
    trend_cache = {}

    for step in range(1, NUM_PASSOS + 1):
        months_elapsed = step * MESES_POR_PASSO
        step_strength = get_step_strength(step, NUM_PASSOS, BASE_STEP_STRENGTH, DECAY_RATE, MIN_RELATIVE_GROWTH)

        total_proc, total_changed, total_kept, total_skipped = 0, 0, 0, 0

        for worksheet in wb.worksheets:
            sheet_title_lower = worksheet.title.strip().lower()
            proc_s, changed_s, kept_s, skipped_s = 0, 0, 0, 0

            for row in range(2, worksheet.max_row + 1):
                anomaly_type = normalize_anomaly_type(worksheet[f"C{row}"].value)
                polygon_string = worksheet[f"E{row}"].value
                area_cm2_value = worksheet[f"F{row}"].value
                image_path_excel = worksheet[f"G{row}"].value

                if not polygon_string:
                    skipped_s += 1
                    total_skipped += 1
                    continue

                current_points = parse_polygon_string(polygon_string)
                if len(current_points) < 3:
                    skipped_s += 1
                    total_skipped += 1
                    continue

                # Força o processamento se a folha for corrosion ou o tipo indicado na linha o for
                is_corrosion = ("corrosion" in sheet_title_lower) or (anomaly_type == "corrosion") or (anomaly_type == "" and "damage" not in sheet_title_lower and "water" not in sheet_title_lower)

                if not is_corrosion or "damage" in sheet_title_lower or "water" in sheet_title_lower or anomaly_type in ["damage", "water"]:
                    kept_s += 1
                    total_kept += 1
                    continue

                image_size = find_matching_image_size(image_path_excel, image_index, size_cache)
                if image_size is None:
                    # Se mesmo assim falhar a correspondência exata, usa um tamanho dinâmico seguro baseado no polígono máximo para não saltar
                    max_x = max(p[0] for p in current_points)
                    max_y = max(p[1] for p in current_points)
                    image_size = (int(max_x) + 200, int(max_y) + 200)

                width, height = image_size
                current_pixel_area = polygon_area_pixels(current_points)
                if current_pixel_area <= 0:
                    skipped_s += 1
                    total_skipped += 1
                    continue

                row_key = (worksheet.title, row)
                if row_key not in trend_cache:
                    trend_cache[row_key] = create_corrosion_trend(current_points, width, height, safe_float(area_cm2_value))

                trend = trend_cache[row_key]
                updated_budget = growth_budget.get(row_key, 0.0) + step_strength
                iterations_to_apply = int(math.floor(updated_budget))
                growth_budget[row_key] = updated_budget - iterations_to_apply

                proc_s += 1
                total_proc += 1

                if iterations_to_apply <= 0:
                    continue

                current_mask = polygon_to_mask(current_points, width, height)
                grown_mask = cv2.dilate(current_mask, trend.kernel, iterations=iterations_to_apply)
                if APPLY_MORPH_CLOSE:
                    grown_mask = close_mask(grown_mask, MORPH_CLOSE_KERNEL_SIZE)

                new_points = mask_to_polygon(grown_mask)
                if len(new_points) < 3:
                    skipped_s += 1
                    total_skipped += 1
                    continue

                new_pix_area = polygon_area_pixels(new_points)
                if new_pix_area <= 0:
                    skipped_s += 1
                    total_skipped += 1
                    continue

                new_area_cm2 = (new_pix_area * trend.cm2_per_pixel2) if trend.cm2_per_pixel2 > 0 else 0.0

                worksheet[f"E{row}"] = polygon_to_string(new_points)
                worksheet[f"F{row}"] = round(new_area_cm2, AREA_DECIMALS)

                changed_s += 1
                total_changed += 1

            print(f"--- Folha: {worksheet.title} | Processadas: {proc_s} | Alteradas: {changed_s} | Ignoradas: {skipped_s} ---")

        output_path = output_dir / build_output_filename(FICHEIRO_EXCEL, step, months_elapsed)
        wb.save(output_path)
        print(f"[GUARDADO] {output_path}")

    print("\nPROCESSO CONCLUÍDO COM SUCESSO!")


if __name__ == "__main__":
    main()

DEGRADAÇÃO TEMPORAL ORIENTADA DA CORROSÃO
--- Folha: Anomalies00 | Processadas: 2191 | Alteradas: 2191 | Ignoradas: 0 ---
--- Folha: corrosion | Processadas: 2191 | Alteradas: 2191 | Ignoradas: 1 ---
--- Folha: damage | Processadas: 0 | Alteradas: 0 | Ignoradas: 1 ---
--- Folha: water | Processadas: 0 | Alteradas: 0 | Ignoradas: 1 ---
[GUARDADO] excel_result/degradacao_temporal_excels/trend/metricas_anomalias_faces_malha_degradacao_t01_00a_06m.xlsx
--- Folha: Anomalies00 | Processadas: 2191 | Alteradas: 2191 | Ignoradas: 0 ---
--- Folha: corrosion | Processadas: 2191 | Alteradas: 2191 | Ignoradas: 1 ---
--- Folha: damage | Processadas: 0 | Alteradas: 0 | Ignoradas: 1 ---
--- Folha: water | Processadas: 0 | Alteradas: 0 | Ignoradas: 1 ---
[GUARDADO] excel_result/degradacao_temporal_excels/trend/metricas_anomalias_faces_malha_degradacao_t02_01a_00m.xlsx
--- Folha: Anomalies00 | Processadas: 2191 | Alteradas: 2191 | Ignoradas: 0 ---
--- Folha: corrosion | Processadas: 2191 | Alteradas: 21

## Reconstruir Imagens

In [4]:
# ============================================================
# RECONSTRUIR IMAGENS DE TODOS OS "EVENTOS" (TODOS OS EXCELS)
# OUTPUT POR EVENTO:
#   1) overlay
#   2) comparison (original | overlay)
# ============================================================

import re
from pathlib import Path
from collections import defaultdict

from openpyxl import load_workbook
from PIL import Image, ImageDraw, ImageFont

# ============================================================
# CONFIG
# ============================================================

PASTA_EXCELS_EVENTOS = 'excel_result/degradacao_temporal_excels/trend'
NOVA_PASTA_IMAGENS = "image"
PASTA_SAIDA = "previews_degradacao_temporal"

SUBPASTA_OVERLAY = "overlay"
SUBPASTA_COMPARISON = "comparison"

EXCEL_GLOB = "*.xlsx"
EXTENSOES_VALIDAS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp", ".webp"}

DESENHAR_CONTORNO = False
ESPESSURA_CONTORNO = 2

COMPARISON_GAP = 20
COR_GAP = (255, 255, 255)

CORES_ANOMALIA = {
    "corrosion": (255, 0, 0),   # vermelho puro
    "damage":    (0, 255, 0),   # verde puro
    "water":     (0, 0, 255),   # azul puro
}

COR_DEFAULT = (255, 255, 255)

# ------------------------------------------------------------
# IDs DAS ANOMALIAS
# ------------------------------------------------------------
DESENHAR_IDS = True
FONTE_ID_SIZE = 32
FONTE_ID_PATH = None

COR_TEXTO_ID = (255, 255, 255)
COR_FUNDO_ID = (0, 0, 0)
ID_PADDING = 5
DESENHAR_CAIXA_ID = True


# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def parse_polygon_string(poly_str):
    if poly_str is None:
        return []

    poly_str = str(poly_str).strip()
    if not poly_str:
        return []

    matches = re.findall(
        r"\[\s*(-?\d+(?:\.\d+)?)\s*,\s*(-?\d+(?:\.\d+)?)\s*\]",
        poly_str
    )

    return [(float(x), float(y)) for x, y in matches]


def clip_points(points, width, height):
    clipped = []
    for x, y in points:
        x = min(max(int(round(x)), 0), width - 1)
        y = min(max(int(round(y)), 0), height - 1)
        clipped.append((x, y))
    return clipped


def get_color(anomaly_type):
    if anomaly_type is None:
        return COR_DEFAULT
    key = str(anomaly_type).strip().lower()
    return CORES_ANOMALIA.get(key, COR_DEFAULT)


def extract_id_after_hash(anomaly_id):
    if anomaly_id is None:
        return ""
    s = str(anomaly_id).strip()
    if "#" in s:
        return s.rsplit("#", 1)[1].strip()
    return s


def polygon_centroid(points):
    if not points:
        return None

    if len(points) < 3:
        xs = [p[0] for p in points]
        ys = [p[1] for p in points]
        return sum(xs) / len(xs), sum(ys) / len(ys)

    area_acc = 0.0
    cx_acc = 0.0
    cy_acc = 0.0
    n = len(points)

    for i in range(n):
        x0, y0 = points[i]
        x1, y1 = points[(i + 1) % n]
        cross = x0 * y1 - x1 * y0
        area_acc += cross
        cx_acc += (x0 + x1) * cross
        cy_acc += (y0 + y1) * cross

    signed_area = area_acc / 2.0

    if abs(signed_area) < 1e-9:
        xs = [p[0] for p in points]
        ys = [p[1] for p in points]
        return sum(xs) / len(xs), sum(ys) / len(ys)

    cx = cx_acc / (6.0 * signed_area)
    cy = cy_acc / (6.0 * signed_area)
    return cx, cy


def load_id_font(font_size):
    if FONTE_ID_PATH:
        try:
            return ImageFont.truetype(FONTE_ID_PATH, font_size)
        except Exception:
            pass

    try:
        return ImageFont.truetype("arial.ttf", font_size)
    except:
        return ImageFont.load_default()


def draw_centered_label(draw, image_size, text, center, font):
    if not text or center is None:
        return

    width, height = image_size
    cx, cy = center

    try:
        bbox = draw.textbbox((0, 0), text, font=font)
        text_w = bbox[2] - bbox[0]
        text_h = bbox[3] - bbox[1]
    except:
        text_w, text_h = (10, 10)

    box_w = text_w + 2 * ID_PADDING
    box_h = text_h + 2 * ID_PADDING

    x0 = int(round(cx - box_w / 2))
    y0 = int(round(cy - box_h / 2))

    x0 = max(0, min(x0, max(0, width - box_w)))
    y0 = max(0, min(y0, max(0, height - box_h)))
    x1 = x0 + box_w
    y1 = y0 + box_h

    if DESENHAR_CAIXA_ID:
        draw.rectangle([x0, y0, x1, y1], fill=COR_FUNDO_ID)

    draw.text(
        (x0 + ID_PADDING, y0 + ID_PADDING),
        text,
        fill=COR_TEXTO_ID,
        font=font
    )


def index_images_by_name(root_folder):
    root = Path(root_folder)
    image_map = {}
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in EXTENSOES_VALIDAS:
            image_map[p.name.lower()] = p
    return image_map


def find_image_flexibly(image_path_excel, image_index):
    if not image_path_excel:
        return None

    # Normaliza as barras para o formato universal antes de extrair o nome
    raw_str = str(image_path_excel).strip().replace("\\", "/")

    file_name = Path(raw_str).name.lower()
    stem_lower = Path(raw_str).stem.lower()

    # 1. Tenta correspondência exata pelo nome do ficheiro (ex: figura_569.png)
    if file_name in image_index:
        return image_index[file_name]

    # 2. Tenta correspondência pelo nome base (ex: figura_569)
    for indexed_name, img_path in image_index.items():
        if Path(indexed_name).stem.lower() == stem_lower:
            return img_path

    # 3. Tenta correspondência parcial
    for indexed_name, img_path in image_index.items():
        if stem_lower in indexed_name.lower() or indexed_name.lower() in stem_lower:
            return img_path

    return None


def make_side_by_side(original_img, overlay_img, gap=20, gap_color=(255, 255, 255)):
    original_img = original_img.convert("RGB")
    overlay_img = overlay_img.convert("RGB")

    w1, h1 = original_img.size
    w2, h2 = overlay_img.size

    out_w = w1 + gap + w2
    out_h = max(h1, h2)

    canvas = Image.new("RGB", (out_w, out_h), gap_color)
    canvas.paste(original_img, (0, 0))
    canvas.paste(overlay_img, (w1 + gap, 0))

    return canvas


def sanitize_stem(name):
    return re.sub(r'[\\/*?:"<>|]+', "_", name)


def load_excel_grouped_by_image(excel_path):
    wb = load_workbook(excel_path, data_only=True)
    por_imagem = defaultdict(list)
    total_rows = 0
    total_valid = 0

    for ws in wb.worksheets:
        for row in range(2, ws.max_row + 1):
            total_rows += 1

            anomaly_id = ws[f"A{row}"].value
            anomaly_type = ws[f"C{row}"].value
            polygon_str = ws[f"E{row}"].value
            image_path_excel = ws[f"G{row}"].value

            if not polygon_str or not image_path_excel:
                continue

            points = parse_polygon_string(polygon_str)
            if len(points) < 3:
                continue

            # Extrai estritamente apenas o nome do ficheiro (ex: figura_442.png)
            clean_image_name = Path(str(image_path_excel)).name.lower()

            por_imagem[clean_image_name].append({
                "sheet": ws.title,
                "row": row,
                "anomaly_id": anomaly_id,
                "anomaly_label": extract_id_after_hash(anomaly_id),
                "anomaly_type": anomaly_type,
                "points": points,
                "image_path_excel": image_path_excel
            })

            total_valid += 1

    return por_imagem, total_rows, total_valid


def process_one_event_excel(excel_path, image_index, output_root):
    event_name = sanitize_stem(excel_path.stem)
    event_dir = output_root / event_name
    overlay_dir = event_dir / SUBPASTA_OVERLAY
    comparison_dir = event_dir / SUBPASTA_COMPARISON

    overlay_dir.mkdir(parents=True, exist_ok=True)
    comparison_dir.mkdir(parents=True, exist_ok=True)

    por_imagem, total_rows, total_valid = load_excel_grouped_by_image(excel_path)

    print("\n" + "=" * 70)
    print(f"[EVENTO] {excel_path.name}")
    print("=" * 70)
    print(f"[INFO] Linhas totais no Excel: {total_rows}")
    print(f"[INFO] Polígonos válidos: {total_valid}")
    print(f"[INFO] Imagens com polígonos: {len(por_imagem)}")

    processadas = 0
    faltam_imagens = 0
    id_font = load_id_font(FONTE_ID_SIZE)

    for image_name, registros in por_imagem.items():
        sample_path_excel = registros[0]["image_path_excel"] if registros else image_name
        img_path = find_image_flexibly(sample_path_excel, image_index)

        if img_path is None:
            print(f"[AVISO] Imagem não encontrada na pasta local: {image_name}")
            faltam_imagens += 1
            continue

        original_img = Image.open(img_path).convert("RGB")
        out_img = original_img.copy()
        w, h = out_img.size
        draw = ImageDraw.Draw(out_img)

        labels_to_draw = []

        for reg in registros:
            pts = clip_points(reg["points"], w, h)
            if len(pts) < 3:
                continue

            cor = get_color(reg["anomaly_type"])
            draw.polygon(pts, fill=cor)

            if DESENHAR_CONTORNO:
                draw.line(pts + [pts[0]], fill=cor, width=ESPESSURA_CONTORNO)

            if DESENHAR_IDS:
                centroid = polygon_centroid(pts)
                label = reg.get("anomaly_label", "")
                labels_to_draw.append({
                    "label": label,
                    "centroid": centroid,
                })

        if DESENHAR_IDS:
            for item in labels_to_draw:
                draw_centered_label(
                    draw=draw,
                    image_size=out_img.size,
                    text=item["label"],
                    center=item["centroid"],
                    font=id_font
                )

        stem = Path(img_path).stem

        out_overlay = overlay_dir / f"{stem}_overlay.png"
        out_img.save(out_overlay)

        comparison_img = make_side_by_side(
            original_img=original_img,
            overlay_img=out_img,
            gap=COMPARISON_GAP,
            gap_color=COR_GAP
        )

        out_comparison = comparison_dir / f"{stem}_comparison.png"
        comparison_img.save(out_comparison)

        processadas += 1

    print(f"[OK] Imagens processadas: {processadas}")
    print(f"[OK] Imagens em falta:   {faltam_imagens}")

    return {
        "event_name": event_name,
        "excel_name": excel_path.name,
        "total_rows": total_rows,
        "total_valid": total_valid,
        "num_images": len(por_imagem),
        "processed": processadas,
        "missing": faltam_imagens,
    }


# ============================================================
# EXECUTAR FLUXO PRINCIPAL
# ============================================================

pasta_excels = Path(PASTA_EXCELS_EVENTOS)
nova_pasta_imgs = Path(NOVA_PASTA_IMAGENS)
pasta_saida = Path(PASTA_SAIDA)

if not pasta_excels.exists():
    raise FileNotFoundError(f"Pasta de excels não encontrada: {pasta_excels}")

if not nova_pasta_imgs.exists():
    raise FileNotFoundError(f"Pasta de imagens não encontrada: {nova_pasta_imgs}")

pasta_saida.mkdir(parents=True, exist_ok=True)

image_index = index_images_by_name(nova_pasta_imgs)
excel_files = sorted(pasta_excels.glob(EXCEL_GLOB))

if not excel_files:
    raise FileNotFoundError(f"Não foram encontrados excels em: {pasta_excels}")

summary = []
for excel_path in excel_files:
    result = process_one_event_excel(
        excel_path=excel_path,
        image_index=image_index,
        output_root=pasta_saida
    )
    summary.append(result)

print("\n" + "=" * 70)
print("PROCESSAMENTO TOTAL CONCLUÍDO")
print("=" * 70)
total_events = len(summary)
total_processed_imgs = sum(x["processed"] for x in summary)
total_missing_imgs = sum(x["missing"] for x in summary)

print(f"Eventos processados:        {total_events}")
print(f"Total imagens processadas:  {total_processed_imgs}")
print(f"Total imagens em falta:     {total_missing_imgs}")
print(f"Resultados em:              {pasta_saida}")


[EVENTO] metricas_anomalias_faces_malha_degradacao_t01_00a_06m.xlsx
[INFO] Linhas totais no Excel: 5709
[INFO] Polígonos válidos: 5706
[INFO] Imagens com polígonos: 387
[OK] Imagens processadas: 387
[OK] Imagens em falta:   0

[EVENTO] metricas_anomalias_faces_malha_degradacao_t02_01a_00m.xlsx
[INFO] Linhas totais no Excel: 5709
[INFO] Polígonos válidos: 5706
[INFO] Imagens com polígonos: 387
[OK] Imagens processadas: 387
[OK] Imagens em falta:   0

[EVENTO] metricas_anomalias_faces_malha_degradacao_t03_01a_06m.xlsx
[INFO] Linhas totais no Excel: 5709
[INFO] Polígonos válidos: 5706
[INFO] Imagens com polígonos: 387
[OK] Imagens processadas: 387
[OK] Imagens em falta:   0

[EVENTO] metricas_anomalias_faces_malha_degradacao_t04_02a_00m.xlsx
[INFO] Linhas totais no Excel: 5709
[INFO] Polígonos válidos: 5706
[INFO] Imagens com polígonos: 387
[OK] Imagens processadas: 387
[OK] Imagens em falta:   0

[EVENTO] metricas_anomalias_faces_malha_degradacao_t05_02a_06m.xlsx
[INFO] Linhas totais no